##Real Use Case Pattern
Raw Data → Parquet (Landing Zone)

Clean Data → Delta (Processing Zone)

Analytics → Delta Tables

Exercises

1. Write DataFrame to Parquet
2. Read and display
3. Filter high-value records
4. Write partitioned Parquet
5. Read only one partition
6. Append new data
7. Create SQL table on Parquet
8. Convert to Delta
9. Perform update after conversion
10. Explain difference between Parquet and Delta

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]

columns = ["visit_id","patient_name","city","department","consultation_fee"]

df = spark.createDataFrame(data, columns)

display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
# 1.
df.write \
.mode("overwrite") \
.parquet("/tmp/landing_patient_parquet")

In [0]:
# 2.
landing_df = spark.read.parquet("/tmp/landing_patient_parquet")
display(landing_df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


In [0]:
# 3.
high_value_df = landing_df.filter("consultation_fee > 4000")
display(high_value_df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
# 4.
df.write \
.mode("overwrite") \
.partitionBy("city") \
.parquet("/tmp/partitioned_patient_parquet")

In [0]:
# 5.
spark.read.parquet("/tmp/partitioned_patient_parquet") \
.filter("city = 'Chennai'") \
.show()

+--------+------------+----------+----------------+-------+
|visit_id|patient_name|department|consultation_fee|   city|
+--------+------------+----------+----------------+-------+
|     105|Vikram Singh| Neurology|            7000|Chennai|
+--------+------------+----------+----------------+-------+



In [0]:
# 6.
new_data = [(106,"Ananya Das","Kolkata","Orthopedics",3000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write \
.mode("append") \
.parquet("/tmp/landing_patient_parquet")

In [0]:
%sql
-- 7
CREATE OR REPLACE VIEW patient_parquet_view2 AS
SELECT * FROM parquet.`dbfs:/tmp/landing_patient_parquet`;

In [0]:
%sql
SELECT * FROM patient_parquet_view2;

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
106,Ananya Das,Kolkata,Orthopedics,3000
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


In [0]:
%sql
-- 8
CONVERT TO DELTA parquet.`dbfs:/tmp/landing_patient_parquet`;

In [0]:
%sql
-- 9.
UPDATE delta.`dbfs:/tmp/landing_patient_parquet`
SET consultation_fee = 6500
WHERE visit_id = 101;

num_affected_rows
1


##Parquet vs Delta 
- Parquet is just a file format used to store data efficiently.
- Delta is an advanced layer built on top of Parquet that adds extra features.
- In Parquet, you can only read and write data.
- In Delta, you can update, delete, and merge data easily.
- Parquet does not support transactions, so errors can mess up your data.
- Delta supports ACID transactions, so your data stays safe and consistent.
- Parquet has no version control. Once changed, old data is gone.
- Delta keeps history (Time Travel), so you can go back to previous versions.
- Parquet handles schema changes poorly.
- Delta handles schema evolution smoothly.